# DistilBERT Experiments

This notebook fine-tunes DistilBERT for sentiment analysis using the existing `train` / `val` / `test` split in `data/master_clean.csv`.

It compares three text columns separately:

| Experiment | Text column | Model |
| --- | --- | --- |
| A | `review_text_raw` | `distilbert-base-multilingual-cased` |
| B | `review_text_fixed` | `distilbert-base-multilingual-cased` |
| C | `review_text_translated` | `distilbert-base-uncased` |

The loss is class-weighted so the smaller neutral class matters more during training.

## 1. Imports and Config

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# Set this to True if CUDA gives unknown errors or the screen blacks out.
FORCE_CPU = False
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

from pathlib import Path
import inspect
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "model_training":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "master_clean.csv"
BASE_OUTPUT_DIR = PROJECT_ROOT / "model_training" / "distilbert_outputs"
BASE_MODEL_DIR = PROJECT_ROOT / "model_training" / "distilbert_sentiment_model"

LABEL_COL = "sentiment"
SPLIT_COL = "split"
LABEL_NAMES = ["negative", "neutral", "positive"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_NAMES)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

MAX_LENGTH = 160
SEED = 42
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5

ALL_EXPERIMENTS = {
    "raw_multilingual": {
        "name": "raw_multilingual",
        "text_col": "review_text_raw",
        "model_name": "distilbert-base-multilingual-cased",
    },
    "fixed_multilingual": {
        "name": "fixed_multilingual",
        "text_col": "review_text_fixed",
        "model_name": "distilbert-base-multilingual-cased",
    },
    "translated_english": {
        "name": "translated_english",
        "text_col": "review_text_translated",
        "model_name": "distilbert-base-uncased",
    },
}

# Change this value to train one experiment at a time:
SELECTED_EXPERIMENT = "raw_multilingual"

if SELECTED_EXPERIMENT not in ALL_EXPERIMENTS:
    raise ValueError(f"Unknown experiment: {SELECTED_EXPERIMENT}")

EXPERIMENT = ALL_EXPERIMENTS[SELECTED_EXPERIMENT]


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() and not FORCE_CPU else "cpu"
print(f"Using device: {device}")

c:\Users\pangt\anaconda\envs\nlpbert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 2. Load Data

In [2]:
df = pd.read_csv(DATA_PATH)

required_cols = {LABEL_COL, SPLIT_COL, EXPERIMENT["text_col"]}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()
unknown_labels = sorted(set(df[LABEL_COL]) - set(LABEL_NAMES))
if unknown_labels:
    raise ValueError(f"Unknown labels found: {unknown_labels}")

df["label"] = df[LABEL_COL].map(LABEL2ID)

for exp in ALL_EXPERIMENTS.values():
    col = exp["text_col"]
    df[col] = df[col].fillna("").astype(str).str.strip()

print("Rows:", len(df))
display(df.groupby([SPLIT_COL, LABEL_COL]).size().unstack(fill_value=0).reindex(columns=LABEL_NAMES))
df[["review_text_raw", "review_text_fixed", "review_text_translated", LABEL_COL, SPLIT_COL]].head()

Rows: 612


sentiment,negative,neutral,positive
split,,,
test,29,16,47
train,134,74,220
val,29,15,48


,review_text_raw,review_text_fixed,review_text_translated,sentiment,split
0,barag dh sampai dengn baik comel pawer buyi po...,barang sudah sampai dengan baik comel power bu...,item arrived well cute power good sound even l...,neutral,test
1,Performance:good\n Quality:superb\n \n Small b...,Performance:good Quality:superb Small but powe...,performance good quality superb small powerful...,neutral,val
2,Bluetooth asik disconnect. Guna aux ada bunyi ...,Bluetooth asyik disconnect. Guna aux ada bunyi...,bluetooth keeps disconnecting use aux noise fe...,negative,train
3,Quality:Padan la dgn harga\n \n Charging Cable...,Quality:Padan dengan harga Charging Cable mema...,quality matches price charging cable right hah...,neutral,test
4,Quality:Sound quality is 7/10..\n \n The suppl...,Quality:Sound quality is 7/10.. The supply don...,quality sound quality supply not even bother c...,negative,test


## 3. Dataset, Metrics, and Weighted Trainer

In [3]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = self.class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "neutral_f1": f1_score(labels, preds, labels=[LABEL2ID["neutral"]], average="macro", zero_division=0),
    }

def make_training_args(output_dir):
    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.01,
        "logging_dir": str(output_dir / "logs"),
        "logging_steps": 20,
        "save_strategy": "epoch",
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "report_to": "none",
        "seed": SEED,
    }

    training_arg_names = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in training_arg_names:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    return TrainingArguments(**kwargs)

def class_report_dict(y_true, y_pred):
    return classification_report(
        y_true,
        y_pred,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )

## 4. Train One Experiment

This function trains one model, evaluates validation and test performance, then saves that experiment's model.

In [4]:
def run_experiment(exp):
    set_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    name = exp["name"]
    text_col = exp["text_col"]
    model_name = exp["model_name"]
    print(f"\n=== {name} ===")
    print(f"Text column: {text_col}")
    print(f"Base model : {model_name}")

    exp_df = df[df[text_col] != ""].copy()
    train_df = exp_df[exp_df[SPLIT_COL] == "train"].reset_index(drop=True)
    val_df = exp_df[exp_df[SPLIT_COL] == "val"].reset_index(drop=True)
    test_df = exp_df[exp_df[SPLIT_COL] == "test"].reset_index(drop=True)

    train_labels = train_df["label"].to_numpy()
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1, 2]),
        y=train_labels,
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    print("Class weights:", class_weights.tolist())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(LABEL_NAMES),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    train_dataset = ReviewDataset(train_df[text_col].tolist(), train_df["label"].tolist(), tokenizer)
    val_dataset = ReviewDataset(val_df[text_col].tolist(), val_df["label"].tolist(), tokenizer)
    test_dataset = ReviewDataset(test_df[text_col].tolist(), test_df["label"].tolist(), tokenizer)

    output_dir = BASE_OUTPUT_DIR / name
    model_dir = BASE_MODEL_DIR / name
    training_args = make_training_args(output_dir)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    trainer.train()

    val_metrics = trainer.evaluate(val_dataset)
    test_metrics = trainer.evaluate(test_dataset)
    test_predictions = trainer.predict(test_dataset)
    y_true = test_predictions.label_ids
    y_pred = np.argmax(test_predictions.predictions, axis=1)
    report = class_report_dict(y_true, y_pred)

    model_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
    with open(model_dir / "label_mapping.json", "w", encoding="utf-8") as f:
        json.dump({"label2id": LABEL2ID, "id2label": ID2LABEL}, f, indent=2)

    result = {
        "experiment": name,
        "text_col": text_col,
        "model_name": model_name,
        "val_accuracy": val_metrics["eval_accuracy"],
        "val_macro_f1": val_metrics["eval_macro_f1"],
        "val_neutral_f1": val_metrics["eval_neutral_f1"],
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_macro_f1": test_metrics["eval_macro_f1"],
        "test_neutral_f1": report["neutral"]["f1-score"],
        "model_dir": str(model_dir),
    }

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=[f"actual_{label}" for label in LABEL_NAMES],
        columns=[f"pred_{label}" for label in LABEL_NAMES],
    )

    print("\nTest classification report:")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))
    display(cm)

    del trainer, model, tokenizer, train_dataset, val_dataset, test_dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## 5. Run Selected Experiment

This trains only the experiment named by `SELECTED_EXPERIMENT` in the first code cell. After it finishes, restart the kernel, change `SELECTED_EXPERIMENT`, and run again for the next experiment.


In [5]:
result = run_experiment(EXPERIMENT)
results_df = pd.DataFrame([result])
display(results_df)



=== raw_multilingual ===
Text column: review_text_raw
Base model : distilbert-base-multilingual-cased
Class weights: [1.0646766424179077, 1.9279279708862305, 0.6484848260879517]


c:\Users\pangt\anaconda\envs\nlpbert\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 10%|▉         | 21/212 [00:07<00:34,  5.53it/s]

{'loss': 1.0833, 'grad_norm': 2.5815300941467285, 'learning_rate': 1.8113207547169813e-05, 'epoch': 0.37}


 19%|█▉        | 41/212 [00:11<00:30,  5.65it/s]

{'loss': 1.0575, 'grad_norm': 3.133782148361206, 'learning_rate': 1.6226415094339625e-05, 'epoch': 0.75}


...

{'train_runtime': 59.9051, 'train_samples_per_second': 28.579, 'train_steps_per_second': 3.539, 'train_loss': 0.7773467212353112, 'epoch': 3.96}


100%|██████████| 12/12 [00:00<00:00, 39.78it/s]



Test classification report:
              precision    recall  f1-score   support

    negative       0.62      0.62      0.62        29
     neutral       0.43      0.19      0.26        16
    positive       0.77      0.91      0.83        47

    accuracy                           0.70        92
   macro avg       0.61      0.57      0.57        92
weighted avg       0.66      0.70      0.67        92



,pred_negative,pred_neutral,pred_positive
actual_negative,18,3,8
actual_neutral,8,3,5
actual_positive,3,1,43


,experiment,text_col,model_name,val_accuracy,val_macro_f1,val_neutral_f1,test_accuracy,test_macro_f1,test_neutral_f1,model_dir
0,raw_multilingual,review_text_raw,distilbert-base-multilingual-cased,0.706522,0.620789,0.380952,0.695652,0.57217,0.26087,c:\Users\pangt\Downloads\ecom-reviews-sentimen...


## 6. Selected Model Summary

This shows the metrics and saved folder for the experiment you just trained.


In [6]:
best = results_df.iloc[0]
print("Completed experiment:")
display(best.to_frame().T)
print(f"Saved model folder: {best['model_dir']}")


Completed experiment:


,experiment,text_col,model_name,val_accuracy,val_macro_f1,val_neutral_f1,test_accuracy,test_macro_f1,test_neutral_f1,model_dir
0,raw_multilingual,review_text_raw,distilbert-base-multilingual-cased,0.706522,0.620789,0.380952,0.695652,0.57217,0.26087,c:\Users\pangt\Downloads\ecom-reviews-sentimen...


Saved model folder: c:\Users\pangt\Downloads\ecom-reviews-sentiment-analysis\model_training\distilbert_sentiment_model\raw_multilingual


## 7. Load Best Model and Predict

In [7]:
best_model_dir = Path(best["model_dir"])
best_tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
best_model.to(device)

def predict_sentiment(text):
    inputs = best_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    inputs = {key: value.to(best_model.device) for key, value in inputs.items()}
    best_model.eval()
    with torch.no_grad():
        logits = best_model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return {
        "label": ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "probabilities": {ID2LABEL[i]: float(prob) for i, prob in enumerate(probs)},
    }

predict_sentiment("not good quality and delivery was slow")

{'label': 'negative',
 'confidence': 0.5533461570739746,
 'probabilities': {'negative': 0.5533461570739746,
  'neutral': 0.31781890988349915,
  'positive': 0.12883496284484863}}